# Lecture 15: Dialogue & Chatbot Systems — Answer Key
### NLP Course 2027 — Week 08

---

> **Instructor Answer Key** — Complete worked solutions for all exercises.

**Topics covered:** Rule-based chatbots, intent classification (TF-IDF vs BERT), entity/slot extraction, FAQ retrieval bots.

In [ ]:
# Setup and imports
import re
import random
import numpy as np
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel, pipeline

random.seed(42)
np.random.seed(42)
print('Imports OK')

---

## Exercise 15.1 — Rule-Based University Chatbot (10+ Intents)

**Task:** Implement `UniversityChatbot` with at least 10 intents: greeting, farewell, admissions, courses, grades, library_hours, tuition, financial_aid, campus_map, contact_info.

**Key concept:** Rule-based chatbots use keyword matching (or regex) to detect intent, then randomly sample from a set of canned responses. They are:
- Fast and fully interpretable
- Easy to maintain and audit
- Poor at paraphrase generalization
- Brittle for ambiguous or compound queries

The design uses a priority list of `(keyword_set, response_list)` tuples: the first rule whose keywords overlap with the user's words fires.

In [ ]:
# Exercise 15.1 — Answer
class UniversityChatbot:
    """
    Rule-based chatbot for a university information system.
    10 intents: greeting, farewell, admissions, courses, grades,
    library_hours, tuition, financial_aid, campus_map, contact_info.
    """

    RULES = [
        # (keyword_set, [list of possible responses])
        # --- greeting ---
        (
            {'hello', 'hi', 'hey', 'good', 'morning', 'afternoon', 'evening', 'howdy'},
            [
                'Hello! Welcome to University Info. How can I help you today?',
                'Hi there! I can answer questions about admissions, courses, grades, and more.',
                'Hey! What would you like to know about the university?',
            ]
        ),
        # --- farewell ---
        (
            {'bye', 'goodbye', 'later', 'farewell', 'thanks', 'thank', 'see'},
            [
                'Goodbye! Have a great day.',
                'Thanks for using University Info. See you next time!',
                'Take care! Feel free to come back if you have more questions.',
            ]
        ),
        # --- admissions ---
        (
            {'apply', 'application', 'admission', 'admissions', 'enroll', 'enrollment',
             'register', 'registration', 'accept', 'acceptance', 'requirements'},
            [
                'Applications open September 1 and close March 1 for fall admission. '
                'Visit admissions.university.edu to apply online.',
                'To apply, submit your transcripts, two letters of recommendation, and a personal statement. '
                'The application portal is at apply.university.edu.',
                'Admission requirements vary by program. Visit admissions.university.edu '
                'or call 1-800-UNI-INFO for details.',
            ]
        ),
        # --- courses ---
        (
            {'course', 'courses', 'class', 'classes', 'lecture', 'lectures', 'syllabus',
             'curriculum', 'program', 'programs', 'subject', 'subjects', 'major', 'minor'},
            [
                'You can browse the full course catalog at courses.university.edu. '
                'Courses are organized by department and updated each semester.',
                'Course registration opens two weeks before each semester. '
                'Log in to the student portal at portal.university.edu to register.',
                'We offer over 3,000 courses across 80+ departments. '
                'Check courses.university.edu for availability and prerequisites.',
            ]
        ),
        # --- grades ---
        (
            {'grade', 'grades', 'gpa', 'transcript', 'score', 'scores', 'result',
             'results', 'mark', 'marks', 'academic', 'performance'},
            [
                'You can view your grades in the student portal at portal.university.edu. '
                'Final grades are posted within two weeks of the last exam.',
                'Grade disputes must be submitted within 30 days of posting. '
                'Contact your instructor first, then the department chair if needed.',
                'Official transcripts can be requested at registrar.university.edu. '
                'Allow 5 business days for processing.',
            ]
        ),
        # --- library_hours ---
        (
            {'library', 'libraries', 'open', 'hours', 'close', 'closed', 'reading',
             'books', 'journal', 'journals', 'database', 'databases'},
            [
                'The main library is open Mon–Thu 8am–10pm, Fri 8am–8pm, and Sat–Sun 10am–6pm. '
                'Extended hours apply during finals week.',
                'We have 5 library branches on campus. Hours vary by location — '
                'see library.university.edu for each branch schedule.',
                'Digital library resources (databases, e-journals) are available 24/7 '
                'at library.university.edu with your student login.',
            ]
        ),
        # --- tuition ---
        (
            {'tuition', 'fee', 'fees', 'cost', 'costs', 'price', 'pay', 'payment',
             'expensive', 'afford', 'billing', 'invoice'},
            [
                'Undergraduate tuition is $15,000/semester for in-state students '
                'and $22,000/semester for out-of-state students.',
                'Tuition bills are due the first day of each semester. '
                'Payment plans are available at bursar.university.edu.',
                'For a full breakdown of tuition and fees, visit bursar.university.edu '
                'or contact the Bursar\'s Office at bursar@university.edu.',
            ]
        ),
        # --- financial_aid ---
        (
            {'financial', 'aid', 'scholarship', 'scholarships', 'grant', 'grants',
             'loan', 'loans', 'fafsa', 'fellowship', 'bursary', 'funding', 'stipend'},
            [
                'Financial aid applications open November 1. Complete the FAFSA at '
                'fafsa.ed.gov and submit your university aid application by February 15.',
                'We award over $50M in scholarships annually. '
                'Visit financialaid.university.edu to see available awards and eligibility.',
                'Contact the Financial Aid Office at aid@university.edu or ext. 4400 '
                'for personalized guidance on your aid package.',
            ]
        ),
        # --- campus_map ---
        (
            {'map', 'campus', 'building', 'buildings', 'location', 'directions',
             'where', 'find', 'navigate', 'parking', 'transport', 'bus', 'shuttle'},
            [
                'An interactive campus map is available at map.university.edu. '
                'You can search for any building, parking lot, or facility.',
                'Campus shuttle runs every 15 minutes between the main quad and '
                'the science complex. See map.university.edu for routes and schedules.',
                'Visitor parking is available in Lots A, B, and C. '
                'See map.university.edu/parking for availability.',
            ]
        ),
        # --- contact_info ---
        (
            {'contact', 'email', 'phone', 'call', 'reach', 'office', 'staff',
             'help', 'support', 'assistance', 'speak', 'talk', 'human', 'person'},
            [
                'General inquiries: info@university.edu | Phone: 1-800-UNI-INFO | '
                'Office hours: Mon–Fri 9am–5pm.',
                'Main switchboard: (555) 100-2000. Department directory at '
                'directory.university.edu.',
                'Visit our Welcome Center in Building A, Room 101, '
                'open Mon–Fri 8am–6pm for walk-in assistance.',
            ]
        ),
    ]

    FALLBACK = (
        'I\'m not sure about that. Please contact the university office at '
        'info@university.edu or call 1-800-UNI-INFO.'
    )

    def respond(self, user_input: str) -> str:
        """Match user input against keyword rules; return a response."""
        words = set(re.sub(r'[^\w\s]', '', user_input.lower()).split())
        for keywords, responses in self.RULES:
            if words & keywords:   # set intersection
                return random.choice(responses)
        return self.FALLBACK


# --- Test ---
bot = UniversityChatbot()
test_queries = [
    ('greeting',      'Hello! I need some information.'),
    ('admissions',    'How do I apply for admission?'),
    ('courses',       'What computer science courses are available?'),
    ('grades',        'Where can I check my GPA?'),
    ('library_hours', 'When is the library open on weekends?'),
    ('tuition',       'What is the tuition fee for in-state students?'),
    ('financial_aid', 'Are there any scholarships I can apply for?'),
    ('campus_map',    'Where is the parking lot near the science building?'),
    ('contact_info',  'I need to speak to someone in the admissions office.'),
    ('farewell',      'Thanks for the help, goodbye!'),
    ('fallback',      'What is the meaning of life?'),
]

print(f'{"Intent":<15} | User Query')
print('=' * 80)
for intent, query in test_queries:
    response = bot.respond(query)
    print(f'{intent:<15} | Q: {query}')
    print(f'{"":<15} | A: {response[:80]}...' if len(response) > 80 else f'{"":<15} | A: {response}')
    print()

**Design notes:**
- Keywords are broad sets to handle paraphrases (e.g., 'apply'/'admission'/'enrollment' all map to admissions).
- Randomizing responses avoids robotic repetition.
- A fallback response is essential for graceful degradation.
- **Limitation:** overlapping keywords can cause wrong intent matches (e.g., 'open courses' could match library_hours). Priority ordering and more specific keywords mitigate this.

---

## Exercise 15.2 — Intent Classification: TF-IDF+LR vs BERT Embeddings

**Task:** Train TF-IDF + Logistic Regression baseline. Then extract DistilBERT [CLS] embeddings and train LR on top. Compare accuracy.

**Key concept:** TF-IDF captures lexical similarity; BERT embeddings capture semantic similarity. On small datasets, TF-IDF is often competitive because the vocabulary is domain-specific and patterns are lexical. BERT shines when paraphrase generalization is needed (e.g., 'cancel purchase' should match 'cancel_order' even if 'order' is never said).

In [ ]:
# Exercise 15.2 — Answer

# --- Training data (expanded to give meaningful train/test split) ---
raw_data = [
    # greeting
    ('Hello there!', 'greeting'),
    ('Hi, how are you?', 'greeting'),
    ('Hey! Good morning.', 'greeting'),
    ('Good afternoon!', 'greeting'),
    ('Howdy, what can you do?', 'greeting'),
    # check_order
    ('Where is my order?', 'check_order'),
    ('Track my package', 'check_order'),
    ('What is the status of my purchase?', 'check_order'),
    ('Can I see my order history?', 'check_order'),
    ('I need to track my shipment.', 'check_order'),
    # cancel_order
    ('I want to cancel my order', 'cancel_order'),
    ('Please cancel my purchase', 'cancel_order'),
    ('How do I cancel?', 'cancel_order'),
    ('Stop my order before it ships', 'cancel_order'),
    ('I changed my mind, cancel it please', 'cancel_order'),
    # return_item
    ('I want to return this item', 'return_item'),
    ('How do I get a refund?', 'return_item'),
    ('This product is broken, I want my money back', 'return_item'),
    ('Return policy question', 'return_item'),
    ('I need to send this back', 'return_item'),
    # farewell
    ('Goodbye!', 'farewell'),
    ('See you later', 'farewell'),
    ('Thanks, bye!', 'farewell'),
    ('Have a great day', 'farewell'),
    ('I am done, thank you', 'farewell'),
    # shipping_info
    ('How much does shipping cost?', 'shipping_info'),
    ('When will it arrive?', 'shipping_info'),
    ('What are your delivery options?', 'shipping_info'),
    ('Do you ship internationally?', 'shipping_info'),
    ('Estimated delivery time for standard shipping?', 'shipping_info'),
    # product_inquiry
    ('Do you have this in blue?', 'product_inquiry'),
    ('Is this item in stock?', 'product_inquiry'),
    ('What sizes are available?', 'product_inquiry'),
    ('Can I see product specifications?', 'product_inquiry'),
    ('Does this come with a warranty?', 'product_inquiry'),
]

texts  = [t for t, _ in raw_data]
labels = [l for _, l in raw_data]

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.25, random_state=42, stratify=labels
)
print(f'Train size: {len(X_train)}, Test size: {len(X_test)}')

# ============================================================
# Baseline 1: TF-IDF + Logistic Regression
# ============================================================
tfidf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
    ('clf',   LogisticRegression(max_iter=500, C=1.0, random_state=42)),
])
tfidf_pipeline.fit(X_train, y_train)
tfidf_preds = tfidf_pipeline.predict(X_test)
tfidf_acc = accuracy_score(y_test, tfidf_preds)
print(f'\nTF-IDF + LR accuracy: {tfidf_acc:.4f}')

# ============================================================
# Baseline 2: DistilBERT [CLS] embeddings + Logistic Regression
# ============================================================
BERT_MODEL = 'distilbert-base-uncased'
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert_model     = AutoModel.from_pretrained(BERT_MODEL)
bert_model.eval()

def get_cls_embeddings(texts, tokenizer, model, batch_size=16):
    """Extract [CLS] token embeddings for a list of texts."""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors='pt'
        )
        with torch.no_grad():
            outputs = model(**encoded)
        # [CLS] is the first token; shape: (batch, seq_len, hidden)
        cls_vecs = outputs.last_hidden_state[:, 0, :].numpy()
        all_embeddings.append(cls_vecs)
    return np.vstack(all_embeddings)

print('\nExtracting BERT [CLS] embeddings...')
X_train_bert = get_cls_embeddings(X_train, bert_tokenizer, bert_model)
X_test_bert  = get_cls_embeddings(X_test,  bert_tokenizer, bert_model)
print(f'Embedding shape: {X_train_bert.shape}  (n_samples x hidden_dim)')

bert_lr = LogisticRegression(max_iter=500, C=1.0, random_state=42)
bert_lr.fit(X_train_bert, y_train)
bert_preds = bert_lr.predict(X_test_bert)
bert_acc = accuracy_score(y_test, bert_preds)
print(f'BERT [CLS] + LR accuracy: {bert_acc:.4f}')

# --- Comparison ---
print('\n--- Comparison ---')
print(f'TF-IDF + LR:      {tfidf_acc:.4f}')
print(f'BERT [CLS] + LR:  {bert_acc:.4f}')
print(f'Delta:            {bert_acc - tfidf_acc:+.4f}')
print()
print('Classification report (BERT):')
print(classification_report(y_test, bert_preds))

**Expected results:** On this small 7-intent dataset, TF-IDF often achieves 80–90% and BERT [CLS]+LR achieves 85–95%. The advantage of BERT is larger when the test set contains paraphrases not seen in training.

**For production intent classification**, options include:
1. Fine-tuned BERT (full sequence classification head) — best accuracy
2. BERT [CLS] + LR — good when compute budget is limited
3. TF-IDF + LR — fast, interpretable, good for narrow domains with lots of data

---

## Exercise 15.3 — Entity Extraction: Dates, Names, Course Codes

**Task:** Add patterns for dates, person names, and course codes (e.g., CS101, NLP302, MATH201) to `EnhancedSlotFiller`.

**Key concept:** Regex slot filling is a form of rule-based Named Entity Recognition. Course codes have a predictable structure: 2–6 uppercase letters followed by 3 digits. Dates appear in many formats. Person names are harder (ambiguous with regular words), so we use a heuristic: capitalized words that are not at the start of a sentence and not known stopwords.

In [ ]:
# Exercise 15.3 — Answer
class EnhancedSlotFiller:
    """
    Regex-based entity extraction for university dialogue.
    Extracts: order_id, date, course_code, person_name.
    """

    # Words to exclude from person-name heuristic
    _TITLE_WORDS = {
        'professor', 'prof', 'doctor', 'dr', 'mr', 'mrs', 'ms', 'dean',
        'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday',
        'january', 'february', 'march', 'april', 'may', 'june',
        'july', 'august', 'september', 'october', 'november', 'december',
        'university', 'college', 'department', 'course', 'class', 'exam',
        'i', 'my', 'me', 'we', 'you', 'he', 'she', 'they',
    }

    PATTERNS = {
        # Alphanumeric order/student IDs
        'order_id': r'\bORD-?[A-Z0-9]{4,10}\b',

        # Dates: MM/DD/YYYY, DD-MM-YYYY, Month DD, Month DDth
        'date': (
            r'\b(?:'
            r'(?:\d{1,2}[/-]\d{1,2}[/-]\d{2,4})'          # 01/15/2024  or  15-01-24
            r'|(?:(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|'
            r'Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)'
            r'\s+\d{1,2}(?:st|nd|rd|th)?(?:,?\s+\d{4})?)'  # January 15 or Jan 15, 2024
            r')\b'
        ),

        # Course codes: 2–6 uppercase letters + 3 digits (+ optional letter suffix)
        'course_code': r'\b[A-Z]{2,6}\d{3}[A-Z]?\b',

        # Person name heuristic: "Professor/Dr/Mr/Ms <Title> <Capitalized>"
        # or just pairs/triples of capitalized words not at sentence start
        'person_name': (
            r'(?:Professor|Prof\.?|Dr\.?|Mr\.?|Mrs\.?|Ms\.?)\s+'
            r'(?:[A-Z][a-z]+(?:\s+[A-Z][a-z]+)?)'
        ),
    }

    def extract(self, text: str) -> dict:
        """Extract all slot types from text. Returns dict of slot_name -> list of matches."""
        results = {}
        for slot, pattern in self.PATTERNS.items():
            matches = re.findall(pattern, text)
            if matches:
                results[slot] = matches

        # Additional heuristic: bare capitalized name pairs (FirstName LastName)
        # that are not already captured and not title/month words
        bare_name_pattern = r'(?<![A-Z])([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)(?![a-z])'
        bare_names = re.findall(bare_name_pattern, text)
        filtered = [
            n for n in bare_names
            if n.split()[0].lower() not in self._TITLE_WORDS
            and not re.match(r'^[A-Z]{2,6}\d{3}', n)  # not a course code
        ]
        if filtered:
            existing = results.get('person_name', [])
            # Deduplicate against title-prefixed names already found
            for n in filtered:
                if not any(n in e for e in existing):
                    existing.append(n)
            results['person_name'] = existing

        return results


# --- Tests ---
filler = EnhancedSlotFiller()

test_cases = [
    'I want to enroll in CS301 starting January 15.',
    'Professor Chen said my grade for NLP202 is wrong.',
    'Can John Smith register for MATH101?',
    'Dr. Emily Rodriguez teaches both BIO310 and CHEM201 on 03/15/2025.',
    'My order ORD-ABC1234 was placed on December 3rd, 2024.',
    'Ms. Sarah Johnson needs to withdraw from HIST405A before May 1.',
]

for text in test_cases:
    slots = filler.extract(text)
    print(f'Input: {text}')
    if slots:
        for slot, values in slots.items():
            print(f'  {slot:<15}: {values}')
    else:
        print('  (no slots extracted)')
    print()

**Regex design principles:**
- **Word boundaries (`\b`)** prevent partial matches (e.g., `CS301` inside `ENCS3010`).
- **Non-greedy alternation** handles multiple date formats in one pattern.
- **Title prefixes** (Dr., Prof., Mr.) anchor person-name extraction without false positives.
- For production, replace regex with a fine-tuned NER model (e.g., spaCy `en_core_web_trf`) for robustness.

---

## Exercise 15.4 — FAQ Bot with TF-IDF Cosine Similarity

**Task:** Build `InstitutionFAQBot` with 15+ Q&A pairs using TF-IDF cosine similarity for retrieval.

**Key concept:** This is a retrieval-based QA system. We index all FAQ questions as TF-IDF vectors, then for a user query compute cosine similarity against all indexed questions. The answer with the highest similarity score is returned if it exceeds a threshold.

cosine_similarity(q, d) = (q · d) / (‖q‖ · ‖d‖)

In [ ]:
# Exercise 15.4 — Answer
class InstitutionFAQBot:
    """FAQ chatbot using TF-IDF cosine similarity for retrieval."""

    FAQ = [
        # (question, answer)  — 18 Q&A pairs
        ('What are the application deadlines?',
         'Fall admission applications are due March 1. Spring admission deadline is October 15.'),

        ('How do I reset my student portal password?',
         'Visit help.university.edu or call the IT helpdesk at ext. 5555. You can also reset via email verification.'),

        ('What is the tuition cost for undergraduates?',
         'Undergraduate tuition is $15,000/semester for in-state and $22,000/semester for out-of-state students.'),

        ('How do I apply for financial aid?',
         'Complete the FAFSA at fafsa.ed.gov by February 15. Then submit the university aid form at financialaid.university.edu.'),

        ('When does course registration open?',
         'Course registration opens two weeks before the start of each semester. Log in to portal.university.edu.'),

        ('How do I view my grades?',
         'Grades are available in the student portal at portal.university.edu under the Academics tab.'),

        ('What are the library hours?',
         'Main library hours: Mon–Thu 8am–10pm, Fri 8am–8pm, Sat–Sun 10am–6pm. Extended during finals week.'),

        ('How do I request an official transcript?',
         'Request official transcripts at registrar.university.edu. Allow 5 business days for processing.'),

        ('Where can I find the campus map?',
         'An interactive campus map is at map.university.edu. You can search for buildings, parking, and facilities.'),

        ('What scholarships are available?',
         'We offer merit and need-based scholarships. See all opportunities at financialaid.university.edu/scholarships.'),

        ('How do I drop a course?',
         'Log in to the student portal and navigate to Registration > Drop Course. Deadlines vary by semester.'),

        ('What is the academic calendar?',
         'The full academic calendar including holidays, exam periods, and registration dates is at registrar.university.edu/calendar.'),

        ('How do I contact student health services?',
         'Student Health is located in Building D, Room 200. Hours: Mon–Fri 8am–5pm. Phone: ext. 6600.'),

        ('Where can I find housing information?',
         'On-campus housing applications open April 1. Visit housing.university.edu for residence hall options and pricing.'),

        ('How do I join student clubs and organizations?',
         'Browse 200+ student organizations at studentlife.university.edu. Club fair is held during the first week of each semester.'),

        ('What dining options are on campus?',
         'We have 8 dining halls and 15 cafes across campus. Meal plan options are at dining.university.edu.'),

        ('How do I appeal a grade?',
         'Grade appeals must be submitted within 30 days of grade posting. Start with your instructor, then the department chair.'),

        ('What are the graduation requirements?',
         'Graduation requires 120 credit hours, a 2.0 GPA, and completion of general education requirements. See registrar.university.edu/graduation.'),
    ]

    def __init__(self):
        questions, self.answers = zip(*self.FAQ)
        # Fit TF-IDF on all questions (bigrams help with phrase matching)
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
        self.faq_matrix = self.vectorizer.fit_transform(questions)

    def respond(self, query: str, threshold: float = 0.1) -> tuple:
        """
        Find the most similar FAQ entry.
        Returns (answer, similarity_score).
        """
        query_vec = self.vectorizer.transform([query])
        sims = cosine_similarity(query_vec, self.faq_matrix).flatten()
        best_idx = sims.argmax()

        if sims[best_idx] >= threshold:
            return self.answers[best_idx], float(sims[best_idx])
        return 'I could not find a matching answer. Please contact support at info@university.edu.', 0.0


# --- Test ---
faq_bot = InstitutionFAQBot()

test_queries = [
    'When is the application due?',
    'I forgot my password for the student portal',
    'How much is tuition per semester?',
    'Can I get a scholarship?',
    'Library schedule on Saturday',
    'How do I sign up for clubs?',
    'I want to appeal my final exam grade',
    'What courses do I need to graduate?',
    'Is there a bus that goes around campus?',         # should return low-confidence or map answer
    'What sports teams does the university have?',    # no matching FAQ
]

print(f'{'Query':<45} | {'Score':>6} | Answer (truncated)')
print('=' * 110)
for q in test_queries:
    answer, score = faq_bot.respond(q)
    print(f'{q:<45} | {score:>6.3f} | {answer[:55]}...' if len(answer) > 55 else f'{q:<45} | {score:>6.3f} | {answer}')

**Retrieval-based FAQ trade-offs:**
- TF-IDF bigrams improve matching for multi-word terms ('application deadline', 'student portal').
- Threshold avoids confidently answering completely unrelated queries.
- **Upgrade path:** Replace TF-IDF with sentence-transformers (`all-MiniLM-L6-v2`) for semantic similarity — handles paraphrase and synonym variation much better.

---

## Summary — Key Concepts

| Exercise | Topic | Key Takeaway |
|----------|-------|--------------|
| 15.1 | Rule-based chatbot | Keyword sets + response lists; fast but not generalizable; priority ordering matters |
| 15.2 | Intent classification | TF-IDF+LR is a strong baseline; BERT [CLS] embeddings add semantic generalization |
| 15.3 | Slot/entity extraction | Regex patterns for structured entities (dates, codes, names); word boundary anchors are critical |
| 15.4 | FAQ retrieval bot | TF-IDF cosine similarity for Q&A retrieval; threshold prevents low-confidence answers |

**Chatbot architecture ladder:**
```
Rule-based (regex/keywords)
  → Retrieval-based (TF-IDF / sentence-transformers)
    → Fine-tuned intent classifier + slot filler
      → End-to-end generative (GPT, T5, LLaMA)
```

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**